In [ ]:
import h5py
import os
import numpy as np

In [ ]:
import h5py
import numpy as np
import os

# --- CONFIGURATION ---
BASE_DIR = "/home/STORAGE/TNG50-4/groups_025"
SNAP_NUM = "025"

# Path to the first file (to read header info)
first_file = os.path.join(BASE_DIR, f"fof_subhalo_tab_{SNAP_NUM}.0.hdf5")

print(f"--> Reading Header from: {first_file}")

# 1. GET PARAMETERS FROM HEADER
with h5py.File(first_file, 'r') as f:
    header = f['Header'].attrs
    num_files = int(header['NumFiles'])      # Total number of chunks (0 to N)
    h_param = header['HubbleParam']          # Little h (e.g., 0.6774)
    scale_factor = header['Time']            # Scale factor 'a' (1/(1+z))
    
    print(f"   NumFiles: {num_files}")
    print(f"   Hubble (h): {h_param}")
    print(f"   Scale Factor (a): {scale_factor}")

# 2. LOOP OVER ALL FILES TO COLLECT RAW R200
# TNG splits data across multiple files. We must read them all.
raw_r200_list = []

print(f"--> Iterating over {num_files} files...")

for i in range(num_files):
    # Construct filename for current chunk
    filename = os.path.join(BASE_DIR, f"fof_subhalo_tab_{SNAP_NUM}.{i}.hdf5")
    
    with h5py.File(filename, 'r') as f:
        # Check if this file actually contains groups
        if 'Group' in f and 'Group_R_Crit200' in f['Group']:
            # Read the dataset
            chunk_data = f['Group']['Group_R_Crit200'][:]
            raw_r200_list.append(chunk_data)

# 3. CONCATENATE AND CONVERT UNITS
if len(raw_r200_list) > 0:
    # Merge list of arrays into one big numpy array
    all_r200_code = np.concatenate(raw_r200_list)
    
    # --- UNIT CONVERSION ---
    # Native TNG Unit: ckpc/h (Comoving kiloparsecs / h)
    # Target Unit:     Mpc (Physical Megaparsecs)
    
    # Formula: Value * (scale_factor / h) / 1000
    all_r200_mpc = (all_r200_code / 1000.0) * scale_factor / h_param
    
    print("-" * 30)
    print(f"RESULTS (Physical Mpc):")
    print(f"Total Halos Found: {len(all_r200_mpc)}")
    print(f"Min R200:  {np.min(all_r200_mpc):.6f} Mpc")
    print(f"Max R200:  {np.max(all_r200_mpc):.6f} Mpc")
    print(f"Mean R200: {np.mean(all_r200_mpc):.6f} Mpc")
    print("-" * 30)
    
    # Example: Print first 10
    print(f"First 10 values: {all_r200_mpc[:10]}")

else:
    print("[ERROR] No R200 data found in any file.")

In [ ]:
np.array(np.where(all_r200_mpc == 0)).shape